# 2. Behavioral analysis (CUE and CUEP)

Reproduces the behavioral panels of **Supplementary Fig. 4**: CUE in panels a-c,
CUEP in panels d-f. For each task prefix, three panels are rendered inline:

1. Psychometric curve (choice) vs coherence level
2. Reaction time vs coherence level
3. UR proportion vs coherence level

For CUEP, an additional "No Ext." comparison curve (labeled as in the paper) is
drawn from the `*_never_ur.pkl` files — the control condition in which the passive
agent never emits the uncertainty response, so no external UR cue is shown; for
CUE no such data exists.

**Prerequisite:** run `1_generate_data.ipynb` first so that `data/` contains the
per-part pickle files (`CUE_part*.pkl`, `CUEP_part*.pkl`, `CUEP_part*_never_ur.pkl`).
Figures are displayed inline only; nothing is saved to disk.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import seaborn as sns
import statsmodels.api as sm

df_pkl_kwargs = {"compression": {"method": "gzip", "compresslevel": 4, "mtime": 1}}


In [ ]:
def order_legend(ax):
    """Fix the legend order to match the paper figures."""
    handles, labels = ax.get_legend_handles_labels()
    ordered = [(h, l) for l in ['PD', 'UR','PD after UR', 'No Ext.'] for h, lbl in zip(handles, labels) if lbl == l]
    if ordered:
        leg = ax.legend(handles=[x[0] for x in ordered], labels=[x[1] for x in ordered])
        for line in leg.get_lines():
            line.set_marker('')

def plot_psychometric_curve(ax, data, x, y, label, color, marker='o'):
    # Logistic fit of the binary outcome (choice) against signed coherence
    _df = data[[x, y]].copy().dropna()
    _df[y+'_bin'] = _df[y].round().astype(int)
    glm_binom = sm.GLM(_df[y+'_bin'], sm.add_constant(_df[x]), family=sm.families.Binomial())
    res = glm_binom.fit()
    # Smooth curve over the observed coherence range
    x_vals = np.linspace(_df[x].min(), _df[x].max(), 200)
    x_pred = sm.add_constant(x_vals)
    y_pred = res.predict(x_pred)
    ax.plot(x_vals, y_pred*100, color=color, label=f"{label}", alpha=0.7)
    # Psychometric dots: mean +/- SEM at each coherence level
    grp = _df.groupby(x)[y+'_bin'].agg(['mean', 'count', 'std']).reset_index()
    grp['se'] = grp['std'] / np.sqrt(grp['count'])
    ax.errorbar(grp[x], grp['mean']*100, yerr=grp['se']*100, fmt=marker, color=color,
                markersize=3.5, linestyle="None",alpha=0.7)


In [ ]:
for cue_prefix in ['CUE', 'CUEP']:
    # Load data with sorted globs
    if cue_prefix == 'CUEP':
        paths = [p for p in sorted(Path('data').glob('CUEP_part*.pkl')) if '_never_ur' not in p.name]
        paths_never_ur = sorted(Path('data').glob('CUEP_part*_never_ur.pkl'))
    else:
        paths = sorted(Path('data').glob('CUE_part*.pkl'))
        paths_never_ur = []
    dfs = [pd.read_pickle(p, **df_pkl_kwargs) for p in paths]
    df = pd.concat(dfs, ignore_index=True)
    del dfs
    # choice: 0/1 = chosen direction 1/2; flip so that 1 = the positive-coherence
    # direction (ground_truth == 1), making the psychometric curve rise with
    # signed coherence
    df['choice'] = 1 - df['choice']

    if cue_prefix=='CUEP':
        # Load all never-UR part files (as generated in 1_generate_data.ipynb)
        dfs_nour = [pd.read_pickle(p, **df_pkl_kwargs) for p in paths_never_ur]
        if dfs_nour:
            df_nour = pd.concat(dfs_nour, ignore_index=True)
            df_nour['choice']=1-df_nour['choice']
        else:
            print("Warning: no CUEP *_never_ur.pkl files found in data/")
            df_nour = None
    else:
        df_nour = None

    dfu = df.query('intoStage2==True')
    dfp = df.query('intoStage2==False')

    # 1. Psychometric Curve (Accuracy) vs Coherence Level
    fig, ax = plt.subplots(figsize=(2.7,2), dpi=300)
    plot_psychometric_curve(ax, dfp, x='dircoh', y='choice', label='PD', color='#0000FF')
    plot_psychometric_curve(ax, dfu, x='dircoh', y='choice', label='PD after UR', color='#FF0000')
    if cue_prefix == "CUEP":
        plot_psychometric_curve(ax, df_nour, x='dircoh', y='choice', label='No Ext.', color='#2CA02C')
    ax.set(xlabel='Coherence (%)', ylabel='Proportion choices (%)', ylim=[-5, 105])
    order_legend(ax)
    fig.tight_layout()
    plt.show()

    # 2. Reaction Time vs Coherence Level
    fig, ax = plt.subplots(figsize=(2.7,2), dpi=300)
    sns.lineplot(data=dfp, x='dircoh', y='rt', errorbar='se', estimator='mean', ax=ax,
                 label='PD', err_style='bars', marker='o', markersize=5, color='#0000FF', alpha=0.7)
    sns.lineplot(data=dfu, x='dircoh', y='rt', errorbar='se', estimator='mean', ax=ax,
                 label='UR', err_style='bars', marker='o', markersize=5, color='#FF7F0E', alpha=0.7)
    if cue_prefix=='CUEP':
        sns.lineplot(data=df_nour, x='dircoh', y='rt', errorbar='se', estimator='mean', ax=ax,
             label='No Ext.', err_style='bars', marker='o', markersize=5, color='#2CA02C', alpha=0.7)
    ax.set(xlabel='Coherence (%)', ylabel='Reaction time (ms)')
    order_legend(ax)
    fig.tight_layout()
    plt.show()

    # 3. UR proportion vs Coherence Level
    df['intoStage2']=df['intoStage2']*100
    fig, ax = plt.subplots(figsize=(2.7,2), dpi=300)
    sns.lineplot(data=df, x='dircoh', y='intoStage2', errorbar='se', estimator='mean', ax=ax,
                 err_style='bars', marker='o', markersize=5, color='#FF7F0E', alpha=0.7,label='UR')
    if cue_prefix=='CUEP':
        df_nour['intoStage2']=df_nour['intoStage2']*100
        sns.lineplot(data=df_nour, x='dircoh', y='intoStage2', errorbar='se', estimator='mean', ax=ax,
             label='No Ext.', err_style='bars', marker='o', markersize=5, color='#2CA02C', alpha=0.7)
        df_nour['intoStage2']=df_nour['intoStage2']/100
    ax.set(xlabel='Coherence (%)', ylabel='UR Proportion (%)')
    ax.legend()
    fig.tight_layout()
    plt.show()
    df['intoStage2']=df['intoStage2']/100

